# Chapter 21 — Did the Bridge Preserve the Space?

**Book alignment:** Embeddings From First Principles, Chapter 21

**Question this notebook isolates:** A bridge yields a *profile*, not a verdict. On RELATE
(Wave 3): does a fitted `MiniLM-L6 → mpnet-base` bridge *invert* the paraphrase-vs-negation
gap (from +0.033 native to −0.107 bridged)? Can a *supervised* bridge exceed the source
encoder's own hard-negative accuracy (so "ceiling" was the wrong word)? And how much does an
A→B→A round trip lose?

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    return json.loads((EXP / wave / "artifacts" / f"{name}.json").read_text())

## 1. Per-relation preservation: the bridge inverts the polarity distinction (Wave 3)

In [ ]:
rp = art("wave3", "relation-preservation")["pairs"]["minilm-l6 vs mpnet-base"]
print(f"paraphrase - negation gap:  native {rp['paraphrase_vs_negation_native']:+.3f}"
      f"   bridged {rp['paraphrase_vs_negation_bridged']:+.3f}")
delta = rp["bridged_minus_native_cosine_by_relation"]
print("bridged - native cosine, by relation:")
for r, dv in sorted(delta.items(), key=lambda kv: kv[1]):
    print(f"  {r:20} {dv:+.3f}")

assert rp["paraphrase_vs_negation_native"] > 0 > rp["paraphrase_vs_negation_bridged"]   # SIGN FLIP
assert delta["negation"] > 0 and delta["paraphrase"] < 0    # pulled negation IN, pushed paraphrase OUT
print("\nthe map did not just weaken the polarity distinction - it reversed it")

## 2. The source-native score is a reference point, not a ceiling (Wave 3)

In [ ]:
sb = art("wave3", "supervised-bridge-ceiling")["pairs"]["minilm-l6 vs mpnet-base"]
print("structured hard-negative accuracy on held-out entities:")
print(f"  source encoder native : {sb['source_native_accuracy']:.3f}")
print(f"  supervised bridge     : {sb['supervised_bridge_accuracy']:.3f}")
assert sb["bridge_exceeds_native"] and sb["supervised_bridge_accuracy"] > sb["source_native_accuracy"]
print("\na contrastive bridge exposed a distinction present-but-cosine-hidden in the source vectors")
print("-> treat the source-native score as a REFERENCE, not a proven maximum")

## 3. Round trips lose a little each way (Wave 3)

In [ ]:
rt = art("wave3", "roundtrip")["pairs"]["mpnet-base <-> bge-large"]
print(f"single forward hop A->B : cosine {rt['single_hop_forward_cosine']:.3f}")
print(f"round trip A->B->A      : cosine {rt['roundtrip_cosine']:.3f}")
assert rt["roundtrip_cosine"] < rt["single_hop_forward_cosine"]
print("composition is lossy; every extra hop compounds it")

## What we earned

A bridge produces a *profile* of eight-plus preservation metrics, not one number. On RELATE
no bridge won every column. A fitted linear bridge *inverted* the paraphrase-vs-negation
gap. A supervised contrastive bridge *exceeded* the source-native hard-negative score — so
that score is a reference point, not a proven ceiling. Calibration rarely transfers.

**Notebook 22 / Chapter 22** asks the same "what survived?" question of a different
transformation: compressing a document to a summary.